# App Product Interest Analysis

상품 매칭 및 비정상 사용자 제거가 끝난 앱 이벤트를 바탕으로, 상품 관심도 분석 결과를 단계별로 확인하는 노트북입니다.

- 입력 데이터: `outputs/cleaned_app_events/clean_app_item_interactions.parquet`
- 분석 모듈: `app_product_interest_analysis.py`
- 결과 위치: `outputs/product_interest_analysis/`
- 주의: 현재 매칭된 상품 이벤트에는 구매 이벤트가 없으므로, 아래 지표는 매출 전환이 아니라 앱 내 관심도와 구매 의도 proxy입니다.

## 0. 환경 준비

분석 모듈을 불러오고 결과 저장 폴더를 준비합니다. 이 셀은 데이터 파일을 수정하지 않고, EDA 출력 폴더만 생성합니다.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

BASE_DIR = Path.cwd()
if not (BASE_DIR / "app_product_interest_analysis.py").exists():
    BASE_DIR = Path.cwd() / "eda" / "app_event_yumi"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

analysis = importlib.import_module("app_product_interest_analysis")
analysis = importlib.reload(analysis)

OUTPUT_DIR = analysis.OUTPUT_DIR
MONTH_DIR = analysis.MONTH_DIR
PERIOD_DIR = analysis.PERIOD_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MONTH_DIR.mkdir(parents=True, exist_ok=True)
PERIOD_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font="AppleGothic")
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR, analysis.INPUT_FILE, OUTPUT_DIR

## 1. 데이터 로드 확인

클린 상품 이벤트 parquet을 LazyFrame으로 연결하고, 컬럼 스키마를 먼저 확인합니다.

In [ ]:
lf = analysis.app_lf()
lf.collect_schema()

## 2. 전체 개요

분석 대상 이벤트 수, 상품 수, 사용자 수, 카테고리 수, 관측 기간을 확인합니다.

In [ ]:
overview = lf.select(
    [
        analysis.pl.len().alias("clean_product_event_rows"),
        analysis.pl.col("mapped_pos_item_code").n_unique().alias("unique_products"),
        analysis.pl.col("af_customer_user_id").n_unique().alias("unique_users"),
        analysis.pl.col("category_l").n_unique().alias("unique_large_categories"),
        analysis.pl.col("event_date").min().alias("start_date"),
        analysis.pl.col("event_date").max().alias("end_date"),
    ]
).collect()

analysis.write_csv(overview, "product_analysis_overview.csv")
overview

## 3. 상품 이벤트 믹스

`view`, `cart`, `wishlist`, `share`, `remove` 이벤트의 비중입니다. 구매 이벤트가 없으므로 장바구니와 위시리스트를 구매 의도 proxy로 봅니다.

In [ ]:
event_mix = (
    lf.group_by("Event Name")
    .agg(analysis.pl.len().alias("event_count"))
    .with_columns(
        (analysis.pl.col("event_count") / analysis.pl.col("event_count").sum() * 100)
        .round(4)
        .alias("event_share_pct")
    )
    .sort("event_count", descending=True)
    .collect()
)

analysis.write_csv(event_mix, "product_event_mix.csv")
event_mix

## 4. 상품별 관심도 지표

`intent_score`는 조회, 장바구니, 위시리스트, 공유를 가중 합산하고 삭제 이벤트를 차감한 앱 관심도 지표입니다.

In [ ]:
products = analysis.product_metrics(lf)
analysis.write_csv(products, "product_interest_metrics.csv")
analysis.write_csv(products.head(200), "top200_products_by_intent_score.csv")

product_cols = [
    "mapped_pos_item_code",
    "mapped_item_name",
    "mapped_category_hierarchy",
    "unique_users",
    "view_count",
    "cart_add_count",
    "wishlist_add_count",
    "share_count",
    "intent_score",
    "cart_add_per_100_views",
    "intent_score_per_user",
]
products.select(product_cols).head(30)

In [ ]:
analysis.save_barplot(
    products.head(20),
    x="intent_score",
    y="mapped_item_name",
    title="Top 20 Products by App Intent Score",
    xlabel="Intent score",
    ylabel="Product",
    filename="top20_products_by_intent_score.png",
)

display(Image(filename=str(OUTPUT_DIR / "top20_products_by_intent_score.png")))

## 5. 장바구니 전환율 상위 상품

조회 수가 최소 500 이상인 상품 중 `cart_add_per_100_views`가 높은 상품입니다. 관심 규모보다 구매 의도 밀도를 확인할 때 사용합니다.

In [ ]:
cart_rate = (
    products.sort("cart_add_per_100_views", descending=True)
    .filter(analysis.pl.col("view_count") >= 500)
    .head(200)
)
analysis.write_csv(cart_rate, "top200_products_by_cart_rate_min500views.csv")
cart_rate.select(product_cols).head(30)

## 6. 사용자 도달 상위 상품

`unique_users`가 큰 상품은 단순 관심 점수와 별개로 앱에서 넓게 노출되었거나 많은 사용자가 반응한 상품입니다.

In [ ]:
user_reach = products.sort("unique_users", descending=True).head(200)
analysis.write_csv(user_reach, "top200_products_by_unique_users.csv")
user_reach.select(product_cols).head(30)

## 7. 카테고리별 관심도

대분류/중분류 단위로 앱 관심도, 상품 수, 사용자 수, 장바구니 전환율을 확인합니다.

In [ ]:
categories = analysis.category_metrics(lf)
analysis.write_csv(categories, "category_interest_metrics.csv")

category_cols = [
    "category_l",
    "category_m",
    "total_events",
    "unique_products",
    "unique_users",
    "weighted_interest_score",
    "view_count",
    "cart_add_count",
    "wishlist_add_count",
    "cart_add_per_100_views",
    "intent_actions_per_100_views",
    "score_per_product",
]
categories.select(category_cols).head(30)

In [ ]:
analysis.save_barplot(
    categories.head(20),
    x="weighted_interest_score",
    y="category_m",
    title="Top 20 Middle Categories by Weighted App Interest",
    xlabel="Weighted interest score",
    ylabel="Middle category",
    filename="top20_middle_categories_by_interest.png",
)

display(Image(filename=str(OUTPUT_DIR / "top20_middle_categories_by_interest.png")))

## 8. 월별 상품 관심도

월별로 앱 관심도 상위 상품을 확인합니다. 시즌성, 캠페인성, 신제품 후보를 나눠 보기 위한 기본 테이블입니다.

In [ ]:
monthly = analysis.monthly_product_metrics(lf)
analysis.write_csv(monthly, "monthly_product_interest_metrics.csv")
analysis.write_csv(monthly.group_by("event_month").head(50), "monthly_top50_products_by_interest.csv")

monthly_cols = [
    "event_month",
    "mapped_pos_item_code",
    "mapped_item_name",
    "category_l",
    "category_m",
    "unique_users",
    "view_count",
    "cart_add_count",
    "wishlist_add_count",
    "weighted_interest_score",
    "cart_add_per_100_views",
]
monthly.select(monthly_cols).head(60)

## 9. 월별 급상승 상품

전월 대비 `weighted_interest_score`가 크게 증가한 상품입니다. 트렌드 후보, 이벤트 반응 상품, IP/콜라보 상품 탐색에 적합합니다.

In [ ]:
rising = analysis.monthly_movers(monthly)
analysis.write_csv(rising.group_by("event_month").head(100), "monthly_top100_rising_products.csv")

rising_cols = monthly_cols + [
    "prev_weighted_interest_score",
    "score_delta_vs_prev_month",
    "unique_users_delta_vs_prev_month",
]
rising.select(rising_cols).head(60)

## 10. 월별 신규 출현 상품

해당 월에 처음 앱 상품 이벤트가 관측된 상품입니다. 출시, 노출 시작, 이벤트 진입 상품 후보로 볼 수 있습니다.

In [ ]:
first_seen = analysis.first_seen_products(monthly)
analysis.write_csv(first_seen.group_by("first_month").head(100), "monthly_top100_first_seen_products.csv")

first_seen_cols = monthly_cols + ["first_month", "first_event_date_in_month"]
first_seen.select(first_seen_cols).head(60)

## 11. 월별 폴더 결과 저장

월별로 개요, 이벤트 믹스, 상품/카테고리 랭킹, 급상승/신규 출현 상품을 각각 저장합니다.

In [ ]:
analysis.write_by_month_outputs(lf, monthly, rising, first_seen)
print(f"Saved month-separated outputs to: {MONTH_DIR}")
sorted(path.name for path in MONTH_DIR.iterdir() if path.is_dir())

## 12. 초순/중순/하순 구간별 결과 저장

3월부터 5월까지 각 월을 초순, 중순, 하순으로 나눠 같은 상품 관심도 지표를 저장합니다.

In [ ]:
analysis.write_by_period_outputs(lf)
print(f"Saved period-separated outputs to: {PERIOD_DIR}")
sorted(path.name for path in PERIOD_DIR.iterdir() if path.is_dir())

## 13. 결과 파일 확인

전체 분석 결과 폴더에 생성된 주요 CSV/PNG 목록을 확인합니다.

In [ ]:
result_files = sorted(
    path.relative_to(OUTPUT_DIR).as_posix()
    for path in OUTPUT_DIR.rglob("*")
    if path.is_file()
)
result_files[:80], len(result_files)

## 14. 다음 단계 메모

- `monthly_top100_rising_products.csv`: 트렌드/신제품 후보 발굴
- `product_interest_metrics.csv`: 상품 노드 feature 후보
- `category_interest_metrics.csv`: 카테고리 노드 feature 후보
- `clean_user_product_edges.parquet`: 사용자-상품 그래프 edge 후보

POS 판매 데이터와 결합할 때는 `mapped_pos_item_code`와 날짜 단위를 기준으로 앱 관심도와 실제 판매량의 lead-lag 관계를 검증하면 됩니다.